**steps**
1. get data
2. separate features and target
3. split the data
4. preprocess the data
5. create a model
6. train it
7. predict
8. evaluate

In [1]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import os

In [2]:
house_data =pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/train.csv")
house_test_data =pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/test.csv")

In [3]:
house_data.head()

,Id,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,...,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,1,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,2,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,...,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,3,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,4,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,5,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,...,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


In [4]:
x=house_data.drop("SalePrice",axis=1)
y=house_data.SalePrice

In [5]:
train_x,val_x,train_y,val_y=train_test_split(x,y,train_size=0.8,test_size=0.2,random_state=0)

In [6]:
numeric_columns =train_x.select_dtypes(include=["number"]).columns
categorical_columns =train_x.select_dtypes(include=["object"]).columns

In [7]:
#preprocessing numeric numerical data
numeric_transformer =SimpleImputer()

#preprocessing categorical data

categorical_transformer =Pipeline(steps =[("imputer",SimpleImputer(strategy="most_frequent")),
                                          ("ordinal",OrdinalEncoder(handle_unknown ="use_encoded_value",unknown_value=-1))])

#bundle together numerical and categrical data

preprocessor =  ColumnTransformer(transformers =[("num",numeric_transformer,numeric_columns ),
                                                ("cat",categorical_transformer,categorical_columns)])

In [8]:
#define model
house_model= RandomForestRegressor(n_estimators=100,random_state=0)

In [9]:
#bundle together preprocessing and model
my_pipeline =Pipeline(steps=[("preprocessor",preprocessor),("model",house_model)])


In [10]:
my_pipeline.fit(train_x,train_y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', SimpleImputer(),
                                                  Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'Hal...
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                ('model', RandomForestRegressor(random_state=0))])

In [11]:
preds =my_pipeline.predict(val_x)

In [12]:
mean_absolute_error(val_y,preds)

17515.124897260273

In [13]:
my_pipeline.fit(x,y)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', SimpleImputer(),
                                                  Index(['Id', 'MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual',
       'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1',
       'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'Hal...
       'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2',
       'Heating', 'HeatingQC', 'CentralAir', 'Electrical', 'KitchenQual',
       'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual',
       'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature',
       'SaleType', 'SaleCondition'],
      dtype='object'))])),
                ('model', RandomForestRegressor(random_state=0))])

In [14]:
test_preds =my_pipeline.predict(
house_test_data)

In [15]:
output =pd.DataFrame({'Id':house_test_data.Id,'SalePrice':test_preds})

In [16]:
output.to_csv("submission.csv",index=False)